In [1]:
! pip install pandas
! pip install openpyxl

In [2]:
import pandas as pd
import json

In [3]:
xl_path = '../data/xl/dados_metas_atualizados.xlsx'

In [4]:
df = pd.read_excel(xl_path, engine='openpyxl', sheet_name='Temas, Planos e ODS')

In [5]:
df.head()

,Número da Meta,Destaque do Título,Descrição da Meta,Projeção,Indicador,Unnamed: 5,Secretaria,Eixo,Temas,Subprefeituras,ODS,Nº da Meta ODS,Planos Municipais
0,1,Entregar 8 novos parques,"Entregar 8 novos parques, para ampliar a ofert...","Com essa medida, a cidade chegará a 125 parque...",Número de parques entregues.,NaN,SVMA,Universo SP,Natureza e ambiente,Campo Limpo; Casa Verde/Cachoeirinha; Cidade A...,11; 13; 15,11.7; 15.1; 13.1,Plano Diretor Estratégico; Plano Municipal de ...
1,2,Requalificar ou ampliar 25 parques,"Requalificar ou ampliar 25 parques, com a melh...",Serão priorizadas as unidades com indicadores ...,Número de parques revitalizados.,NaN,SVMA,Universo SP,Natureza e ambiente,Butantã; Capão Redondo; Ermelino Matarazzo; It...,11; 13; 15,11.7; 15.1; 13.1,Plano Diretor Estratégico; Plano Municipal de ...
2,3,Realizar o projeto São Paulo +Verde,"Realizar o projeto São Paulo +Verde, que vai p...",A iniciativa envolve o plantio de árvores em p...,Número de árvores plantadas.,NaN,SVMA; SMSUB,Universo SP,Natureza e ambiente,NaN,2; 11; 13; 15,2.5; 11.7; 13.1; 15.1; 15.2,Plano Municipal de Arborização Urbana
3,4,"Alcançar 11,4% de tratamento de materiais reci...","Alcançar 11,4% de tratamento de materiais reci...",O aumento da capacidade de separação de resídu...,Percentual de resíduos recicláveis não destina...,NaN,SP REGULA; SMSUB; SMDET,Universo SP,Natureza e ambiente,NaN,12; 16,12.5; 16.6,Plano de Gestão Integrada de Resíduos Sólidos
4,5,Levar atividades de conscientização e educação...,Levar atividades de conscientização e educação...,Além de cursos e atividades formativas da Univ...,Número de atendimentos em ações de educação am...,NaN,SVMA,Universo SP,Natureza e ambiente,NaN,3; 9; 16,3.6; 9.1; 16.6,Plano Municipal de Educação Ambiental


In [6]:
renomear = {
    'Número da Meta' : 'meta_numero',
    'Planos Municipais' : 'planos_setoriais'
}

In [7]:
df.rename(columns=renomear, inplace=True)

In [8]:
df = df[renomear.values()].copy()

In [9]:
df.head()

,meta_numero,planos_setoriais
0,1,Plano Diretor Estratégico; Plano Municipal de ...
1,2,Plano Diretor Estratégico; Plano Municipal de ...
2,3,Plano Municipal de Arborização Urbana
3,4,Plano de Gestão Integrada de Resíduos Sólidos
4,5,Plano Municipal de Educação Ambiental


In [10]:
df['planos_setoriais'].isnull().mean()

np.float64(0.30303030303030304)

In [11]:
df.dropna(subset=['planos_setoriais'], inplace=True)

In [12]:
df['planos_setoriais'].isnull().any()

np.False_

## Atualizar planos setoriais

Não vou partir do pressuposto que eles se mantiveram. Vou zerar a seed e gerar eles de novo a partir da planilha (é uma seed simples)

In [13]:
path_planos_setoriais = '../../cadastros_basicos/data/planos.json'

In [14]:
with open(path_planos_setoriais, 'r', encoding='utf-8') as f:
    planos_setoriais = json.load(f)

Note que nenhum tem descricao

In [15]:
planos_setoriais[:3]

[{'nome': 'Plano Municipal de Promoção da Igualdade Racial', 'descricao': ''},
 {'nome': 'Plano de Gestão Integrada de Resíduos Sólidos', 'descricao': ''},
 {'nome': 'AIU Setor Central', 'descricao': ''}]

In [16]:
planos_dados = set()
for i, row in df.iterrows():
    planos = row['planos_setoriais'].split(';')
    planos = [p.strip() for p in planos if p.strip()]
    planos_dados.update(planos)

In [17]:
planos_dados

{'AIU Setor Central',
 'Operação Urbana Faria Lima',
 'Operação Urbana Água Branca',
 'Operação Urbana Água Espraiada',
 'Planejamento Estratégico da CGM 2024-2026',
 'Plano Cicloviário',
 'Plano Diretor Estratégico',
 'Plano Diretor de Drenagem',
 'Plano Emergencial de Calçadas',
 'Plano Municipal de Agroecologia e Desenvolvimento Rural Sustentável',
 'Plano Municipal de Arborização Urbana',
 'Plano Municipal de Assistência Social',
 'Plano Municipal de Ações para Pessoas com Deficiência',
 'Plano Municipal de Cultura',
 'Plano Municipal de Desenvolvimento Econômico',
 'Plano Municipal de Desestatização',
 'Plano Municipal de Educação',
 'Plano Municipal de Educação Ambiental',
 'Plano Municipal de Esportes e Lazer',
 'Plano Municipal de Mobilidade Urbana',
 'Plano Municipal de Promoção da Igualdade Racial',
 'Plano Municipal de Redução de Riscos',
 'Plano Municipal de Saúde',
 'Plano Municipal de Áreas Protegidas, Áreas Verdes e Espaços Livres',
 'Plano Municipal pela Primeira Infânc

In [18]:
planos_seed = []

for plano in planos_dados:
    planos_seed.append({
        "nome" : plano,
        "descricao" : ""
    })

In [19]:
with open(path_planos_setoriais, 'w', encoding='utf-8') as f:
    json.dump(planos_seed, f, ensure_ascii=False, indent=4)

Agora podemos fazer a seed da relacao entre meta e plano setorial

In [20]:
metas_planos = {}

for i, row in df.iterrows():
    meta_numero = row['meta_numero']
    if meta_numero in metas_planos:
        raise ValueError(f'Meta {meta_numero} duplicada')
    
    planos = row['planos_setoriais'].split(';')
    planos = [p.strip() for p in planos if p.strip()]
    metas_planos[meta_numero] = planos

In [21]:
metas_planos[1]

['Plano Diretor Estratégico',
 'Plano Municipal de Áreas Protegidas, Áreas Verdes e Espaços Livres']

In [22]:
with open('../data/metas_planos.json', 'w', encoding='utf-8') as f:
    json.dump(metas_planos, f, ensure_ascii=False,  indent=4)